# Baselines — FVC2004

Two baselines for fingerprint verification:

1. **NCC** — No learning. Pixel-level normalized cross-correlation score, thresholded to classify pairs.
2. **Siamese ResNet-18** — ImageNet-pretrained ResNet-18 encoder (128-D embeddings, cosine similarity, BCE loss).

Outputs: FAR/FRR plots, model weights, and `baseline_results.json`.

> **Note:** The first run may download ImageNet weights (~45MB). If download fails (e.g. restricted environment), the notebook falls back to random initialization with a warning.

In [1]:
# Imports and path setup
from __future__ import annotations

import csv
import json
import random
import time
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18

_cwd = Path.cwd()
if _cwd.name in ("notebooks", "notebooks_fvc2004"):
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "fvc2004"
FIG_DIR = ARTIFACT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# CPU avoids MPS instability on some setups; set False to try CUDA/MPS
FORCE_CPU = True
if not FORCE_CPU and torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif not FORCE_CPU and getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

IMG_SIZE_NCC = (128, 128)
IMG_SIZE_CNN = (224, 224)

plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight"})
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("Device:", DEVICE)

PROJECT_ROOT: /Users/spartan/Desktop/CS228/Project
ARTIFACT_DIR: /Users/spartan/Desktop/CS228/Project/artifacts/fvc2004
Device: cpu


## Data Loading

In [2]:
# Pair CSV loader and PyTorch dataset with in-memory image caching
def load_pairs_csv(path: Path) -> list[dict]:
    with path.open() as f:
        return list(csv.DictReader(f))


def load_gray(path: Path, size: tuple[int, int]) -> np.ndarray:
    with Image.open(path) as img:
        return np.asarray(img.convert("L").resize(size), dtype=np.float32) / 255.0


class FingerprintPairDataset(Dataset):
    """Verification pairs with in-memory cache. `channels=3` repeats gray for ResNet."""

    def __init__(
        self,
        pairs_csv: Path,
        project_root: Path,
        size: tuple[int, int],
        channels: int = 1,
        max_pairs: int | None = None,
    ):
        self.project_root = project_root
        self.size = size
        self.channels = channels
        self.pairs = load_pairs_csv(pairs_csv)
        if max_pairs and len(self.pairs) > max_pairs:
            random.shuffle(self.pairs)
            self.pairs = self.pairs[:max_pairs]
        self._cache: dict[str, torch.Tensor] = {}
        self._preload()

    def _preload(self) -> None:
        uniq: set[str] = set()
        for p in self.pairs:
            uniq.add(p["image_path_1"])
            uniq.add(p["image_path_2"])
        h, w = self.size[1], self.size[0]
        bytes_per = h * w * 4 * self.channels
        print(f"  Preloading {len(uniq)} unique images ({self.channels} ch, {self.size[0]}×{self.size[1]})...")
        for i, rel in enumerate(sorted(uniq)):
            arr = load_gray(self.project_root / rel, self.size)
            t = torch.from_numpy(arr).unsqueeze(0)
            if self.channels == 3:
                t = t.repeat(3, 1, 1)
            self._cache[rel] = t
            if (i + 1) % 500 == 0:
                print(f"    {i + 1}/{len(uniq)}")
        print(f"  Cache ~{len(self._cache) * bytes_per / 1e6:.0f} MB")

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int):
        p = self.pairs[idx]
        t1 = self._cache[p["image_path_1"]]
        t2 = self._cache[p["image_path_2"]]
        label = int(p["label"])
        return t1, t2, label, p.get("finger_id_1", ""), p.get("finger_id_2", "")


_ds = FingerprintPairDataset(ARTIFACT_DIR / "pairs_test.csv", PROJECT_ROOT, IMG_SIZE_CNN, channels=3, max_pairs=32)
x1, x2, y, _, _ = _ds[0]
print("Sanity batch shapes:", x1.shape, x2.shape, "label", y)

  Preloading 62 unique images (3 ch, 224×224)...
  Cache ~37 MB
Sanity batch shapes: torch.Size([3, 224, 224]) torch.Size([3, 224, 224]) label 0


## Verification Metrics

In [3]:
# Sweep thresholds to compute FAR, FRR, EER, and accuracy; plot FAR vs FRR curve
def compute_verification_metrics(labels: np.ndarray, scores: np.ndarray, num_thresholds: int = 500):
    thresholds = np.linspace(float(scores.min()), float(scores.max()), num_thresholds)
    genuine = labels == 1
    impostor = labels == 0
    n_gen = int(genuine.sum())
    n_imp = int(impostor.sum())

    fars, frrs = [], []
    for t in thresholds:
        predicted_match = scores >= t
        far = float((predicted_match & impostor).sum() / max(n_imp, 1))
        frr = float((~predicted_match & genuine).sum() / max(n_gen, 1))
        fars.append(far)
        frrs.append(frr)

    fars = np.array(fars)
    frrs = np.array(frrs)
    eer_idx = int(np.argmin(np.abs(fars - frrs)))
    eer = float((fars[eer_idx] + frrs[eer_idx]) / 2)
    eer_threshold = float(thresholds[eer_idx])
    preds_at_eer = (scores >= eer_threshold).astype(int)
    accuracy = float(accuracy_score(labels, preds_at_eer))

    return {
        "eer": eer,
        "eer_threshold": eer_threshold,
        "accuracy_at_eer": accuracy,
        "far_at_eer": float(fars[eer_idx]),
        "frr_at_eer": float(frrs[eer_idx]),
        "thresholds": thresholds,
        "fars": fars,
        "frrs": frrs,
    }


def plot_far_frr(metrics: dict, title: str, save_path: Path | None = None):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(metrics["thresholds"], metrics["fars"], label="FAR", color="red")
    ax.plot(metrics["thresholds"], metrics["frrs"], label="FRR", color="blue")
    ax.axvline(metrics["eer_threshold"], ls="--", color="gray", label=f"EER = {metrics['eer']:.4f}")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Error rate")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    plt.show()
    return fig


print("Metrics ready")

Metrics ready


## Baseline 1: NCC (No Learning)

In [4]:
# Compute NCC scores on test pairs and evaluate metrics
NCC_EVAL_CAP = 2500


def ncc_score(a: np.ndarray, b: np.ndarray) -> float:
    a_flat = a.flatten() - a.mean()
    b_flat = b.flatten() - b.mean()
    denom = np.linalg.norm(a_flat) * np.linalg.norm(b_flat)
    if denom < 1e-12:
        return 0.0
    return float(np.dot(a_flat, b_flat) / denom)


test_pairs = load_pairs_csv(ARTIFACT_DIR / "pairs_test.csv")
random.shuffle(test_pairs)
test_pairs_ncc = test_pairs[:NCC_EVAL_CAP]

ncc_scores, ncc_labels = [], []
for i, p in enumerate(test_pairs_ncc):
    img1 = load_gray(PROJECT_ROOT / p["image_path_1"], IMG_SIZE_NCC)
    img2 = load_gray(PROJECT_ROOT / p["image_path_2"], IMG_SIZE_NCC)
    ncc_scores.append(ncc_score(img1, img2))
    ncc_labels.append(int(p["label"]))
    if (i + 1) % 1000 == 0:
        print(f"  NCC: {i + 1}/{len(test_pairs_ncc)}")

ncc_scores = np.array(ncc_scores)
ncc_labels = np.array(ncc_labels)

ncc_metrics = compute_verification_metrics(ncc_labels, ncc_scores)
print(
    f"\nNCC — EER: {ncc_metrics['eer']:.4f}, Acc@EER: {ncc_metrics['accuracy_at_eer']:.4f}, "
    f"FAR@EER: {ncc_metrics['far_at_eer']:.4f}, FRR@EER: {ncc_metrics['frr_at_eer']:.4f}"
)

  NCC: 1000/2500
  NCC: 2000/2500

NCC — EER: 0.3084, Acc@EER: 0.6916, FAR@EER: 0.3073, FRR@EER: 0.3094


In [5]:
# Plot and save FAR vs FRR curve for NCC
plot_far_frr(ncc_metrics, "FAR vs FRR — NCC baseline (FVC2004)", FIG_DIR / "far_frr_ncc_baseline.png")
print("Saved: far_frr_ncc_baseline.png")

Saved: far_frr_ncc_baseline.png


/var/folders/hz/lsbfs9y94sb6m6rhk5nj5bth0000gr/T/ipykernel_60494/2033358938.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Baseline 2: Siamese ResNet-18

Shared ResNet-18 encoder (ImageNet weights, 128-D output). Cosine similarity between embeddings gives a score in [0, 1]. Trained with BCE loss on genuine/impostor pair labels.

In [6]:
# Build Siamese ResNet-18 — shared encoder with cosine similarity head
def build_resnet18_encoder(embedding_dim: int = 128) -> nn.Module:
    try:
        net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        print("ResNet-18: ImageNet weights loaded.")
    except Exception as e:
        print("Warning: could not load ImageNet weights (offline or cache issue).")
        print(" ", e)
        net = resnet18(weights=None)
    feats = net.fc.in_features
    net.fc = nn.Linear(feats, embedding_dim)
    return net


class SiameseVerifier(nn.Module):
    def __init__(self, encoder: nn.Module):
        super().__init__()
        self.encoder = encoder

    def forward(self, img1: torch.Tensor, img2: torch.Tensor) -> torch.Tensor:
        z1 = self.encoder(img1)
        z2 = self.encoder(img2)
        sim = F.cosine_similarity(z1, z2, dim=1)
        return (sim + 1) / 2


encoder = build_resnet18_encoder(embedding_dim=128)
model = SiameseVerifier(encoder).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Siamese ResNet-18 parameters: {total_params:,}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/spartan/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%


ResNet-18: ImageNet weights loaded.
Siamese ResNet-18 parameters: 11,242,176


In [7]:
# Training setup — hyperparameters, data loaders, optimizer, loss
TRAIN_PAIRS_CAP = 8_000
VAL_PAIRS_CAP = 2_000
BATCH_SIZE = 24
EPOCHS = 6
LR = 3e-4

train_ds = FingerprintPairDataset(
    ARTIFACT_DIR / "pairs_train.csv",
    PROJECT_ROOT,
    IMG_SIZE_CNN,
    channels=3,
    max_pairs=TRAIN_PAIRS_CAP,
)
val_ds = FingerprintPairDataset(
    ARTIFACT_DIR / "pairs_val.csv",
    PROJECT_ROOT,
    IMG_SIZE_CNN,
    channels=3,
    max_pairs=VAL_PAIRS_CAP,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCELoss()

print(f"Train pairs: {len(train_ds)}, Val: {len(val_ds)}, batches/epoch: {len(train_loader)}")

  Preloading 2463 unique images (3 ch, 224×224)...
    500/2463
    1000/2463
    1500/2463
    2000/2463
  Cache ~1483 MB
  Preloading 527 unique images (3 ch, 224×224)...
    500/527
  Cache ~317 MB
Train pairs: 8000, Val: 2000, batches/epoch: 334


In [8]:
# Training loop — forward pass, BCE loss, backprop, weight update per epoch
train_losses, val_losses = [], []
n_batches = len(train_loader)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    run = 0.0
    for bi, (img1, img2, labels, _, _) in enumerate(train_loader):
        img1 = img1.to(DEVICE)
        img2 = img2.to(DEVICE)
        labels = labels.float().to(DEVICE)
        scores = model(img1, img2)
        loss = criterion(scores, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        run += loss.item()
        if (bi + 1) % 50 == 0:
            print(f"  Epoch {epoch} batch {bi + 1}/{n_batches} loss={loss.item():.4f}")

    train_losses.append(run / n_batches)

    model.eval()
    vloss = 0.0
    with torch.no_grad():
        for img1, img2, labels, _, _ in val_loader:
            img1 = img1.to(DEVICE)
            img2 = img2.to(DEVICE)
            labels = labels.float().to(DEVICE)
            vloss += criterion(model(img1, img2), labels).item()
    val_losses.append(vloss / len(val_loader))
    print(
        f"Epoch {epoch}/{EPOCHS} train_loss={train_losses[-1]:.4f} val_loss={val_losses[-1]:.4f} "
        f"({time.time() - t0:.1f}s)"
    )

ckpt_path = ARTIFACT_DIR / "siamese_resnet18_baseline.pt"
torch.save(model.state_dict(), ckpt_path)
print("Saved:", ckpt_path)

  Epoch 1 batch 50/334 loss=0.3681
  Epoch 1 batch 100/334 loss=0.7513
  Epoch 1 batch 150/334 loss=0.4627
  Epoch 1 batch 200/334 loss=0.4984
  Epoch 1 batch 250/334 loss=0.4226
  Epoch 1 batch 300/334 loss=0.5496
Epoch 1/6 train_loss=0.5165 val_loss=0.5527 (953.9s)
  Epoch 2 batch 50/334 loss=0.3906
  Epoch 2 batch 100/334 loss=0.5327
  Epoch 2 batch 150/334 loss=0.7767
  Epoch 2 batch 200/334 loss=0.3477
  Epoch 2 batch 250/334 loss=0.6036
  Epoch 2 batch 300/334 loss=0.3240
Epoch 2/6 train_loss=0.5037 val_loss=0.4919 (7151.1s)
  Epoch 3 batch 50/334 loss=0.4039
  Epoch 3 batch 100/334 loss=0.4113
  Epoch 3 batch 150/334 loss=0.1675
  Epoch 3 batch 200/334 loss=0.4191
  Epoch 3 batch 250/334 loss=0.6382
  Epoch 3 batch 300/334 loss=0.3492
Epoch 3/6 train_loss=0.4737 val_loss=0.4993 (983.5s)
  Epoch 4 batch 50/334 loss=0.3745
  Epoch 4 batch 100/334 loss=0.5242
  Epoch 4 batch 150/334 loss=0.2811
  Epoch 4 batch 200/334 loss=0.3715
  Epoch 4 batch 250/334 loss=0.4540
  Epoch 4 batch 

In [9]:
# Plot training and validation loss per epoch
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, EPOCHS + 1), train_losses, marker="o", label="Train")
ax.plot(range(1, EPOCHS + 1), val_losses, marker="s", label="Val")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE loss")
ax.set_title("Siamese ResNet-18 — training curve")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "siamese_resnet18_training_curve.png")
plt.show()
print("Saved: siamese_resnet18_training_curve.png")

Saved: siamese_resnet18_training_curve.png


/var/folders/hz/lsbfs9y94sb6m6rhk5nj5bth0000gr/T/ipykernel_60494/2342215332.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Evaluate on Test Set

In [10]:
# Run Siamese ResNet-18 on test pairs and compute FAR, FRR, EER
TEST_PAIRS_CAP = 2500

test_ds = FingerprintPairDataset(
    ARTIFACT_DIR / "pairs_test.csv",
    PROJECT_ROOT,
    IMG_SIZE_CNN,
    channels=3,
    max_pairs=TEST_PAIRS_CAP,
)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model.eval()
rn_scores, rn_labels = [], []
with torch.no_grad():
    for img1, img2, labels, _, _ in test_loader:
        img1 = img1.to(DEVICE)
        img2 = img2.to(DEVICE)
        rn_scores.extend(model(img1, img2).cpu().numpy().tolist())
        rn_labels.extend(labels.numpy().tolist())

rn_scores = np.array(rn_scores)
rn_labels = np.array(rn_labels)

rn_metrics = compute_verification_metrics(rn_labels, rn_scores)
print(
    f"\nSiamese ResNet-18 — EER: {rn_metrics['eer']:.4f}, Acc@EER: {rn_metrics['accuracy_at_eer']:.4f}, "
    f"FAR@EER: {rn_metrics['far_at_eer']:.4f}, FRR@EER: {rn_metrics['frr_at_eer']:.4f}"
)

  Preloading 528 unique images (3 ch, 224×224)...
    500/528
  Cache ~318 MB

Siamese ResNet-18 — EER: 0.0872, Acc@EER: 0.9128, FAR@EER: 0.0865, FRR@EER: 0.0879


In [11]:
# Plot and save FAR vs FRR curve for Siamese ResNet-18
plot_far_frr(
    rn_metrics,
    "FAR vs FRR — Siamese ResNet-18 (FVC2004)",
    FIG_DIR / "far_frr_siamese_resnet18_baseline.png",
)
print("Saved: far_frr_siamese_resnet18_baseline.png")

Saved: far_frr_siamese_resnet18_baseline.png


/var/folders/hz/lsbfs9y94sb6m6rhk5nj5bth0000gr/T/ipykernel_60494/2033358938.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Error Analysis

In [12]:
# Identify false accepts and false rejects at the EER threshold
threshold = rn_metrics["eer_threshold"]
predicted = (rn_scores >= threshold).astype(int)

false_accepts: list[dict] = []
false_rejects: list[dict] = []
eval_pairs = test_ds.pairs[: len(rn_labels)]

for i, p in enumerate(eval_pairs):
    tl, pl = int(p["label"]), int(predicted[i])
    row = {**p, "score": float(rn_scores[i])}
    if tl == 0 and pl == 1:
        false_accepts.append(row)
    elif tl == 1 and pl == 0:
        false_rejects.append(row)

print(f"Threshold ~EER: {threshold:.4f}")
print(f"False accepts: {len(false_accepts)}  False rejects: {len(false_rejects)}  (n={len(rn_labels)})")

manifest_lookup: dict[str, dict] = {}
for r in load_pairs_csv(ARTIFACT_DIR / "manifest_with_split.csv"):
    manifest_lookup[r["image_path"]] = r


def part_sensor_key(path: str) -> str:
    m = manifest_lookup.get(path, {})
    return f"{m.get('partition', '?')}|{m.get('sensor', '?')}"


if false_rejects:
    keys = [f"{part_sensor_key(fr['image_path_1'])} vs {part_sensor_key(fr['image_path_2'])}" for fr in false_rejects[:300]]
    print("\nFalse reject — partition|sensor combinations (top):")
    for k, v in Counter(keys).most_common(12):
        print(f"  {v:4d}  {k}")

if false_accepts:
    cross = 0
    for fa in false_accepts[:400]:
        p1 = manifest_lookup.get(fa["image_path_1"], {}).get("partition", "")
        p2 = manifest_lookup.get(fa["image_path_2"], {}).get("partition", "")
        if p1 and p2 and p1 != p2:
            cross += 1
    print(
        f"\nFalse accepts with images from different FVC partitions (DB×set): "
        f"{cross}/{min(len(false_accepts), 400)}"
    )

Threshold ~EER: 0.8047
False accepts: 108  False rejects: 110  (n=2500)

False reject — partition|sensor combinations (top):
    33  DB1_A|optical_crossmatch_v300 vs DB1_A|optical_crossmatch_v300
    27  DB4_A|synthetic_sfinge vs DB4_A|synthetic_sfinge
    19  DB2_A|optical_digitalpersona_uareu4000 vs DB2_A|optical_digitalpersona_uareu4000
     8  DB3_A|thermal_atmel_fingerchip vs DB3_A|thermal_atmel_fingerchip
     8  DB4_B|synthetic_sfinge vs DB4_B|synthetic_sfinge
     7  DB2_B|optical_digitalpersona_uareu4000 vs DB2_B|optical_digitalpersona_uareu4000
     6  DB3_B|thermal_atmel_fingerchip vs DB3_B|thermal_atmel_fingerchip
     2  DB1_B|optical_crossmatch_v300 vs DB1_B|optical_crossmatch_v300

False accepts with images from different FVC partitions (DB×set): 48/108


## Save Results

In [13]:
# Save NCC and Siamese ResNet-18 results to baseline_results.json and print comparison table
results = {
    "ncc_baseline": {
        "eer": ncc_metrics["eer"],
        "eer_threshold": ncc_metrics["eer_threshold"],
        "accuracy_at_eer": ncc_metrics["accuracy_at_eer"],
        "far_at_eer": ncc_metrics["far_at_eer"],
        "frr_at_eer": ncc_metrics["frr_at_eer"],
        "test_pairs_evaluated": int(len(ncc_labels)),
    },
    "siamese_resnet18_baseline": {
        "eer": rn_metrics["eer"],
        "eer_threshold": rn_metrics["eer_threshold"],
        "accuracy_at_eer": rn_metrics["accuracy_at_eer"],
        "far_at_eer": rn_metrics["far_at_eer"],
        "frr_at_eer": rn_metrics["frr_at_eer"],
        "test_pairs_evaluated": int(len(rn_labels)),
        "model_params": total_params,
        "epochs_trained": EPOCHS,
        "train_pairs_used": len(train_ds),
        "backbone": "resnet18_imagenet",
        "embedding_dim": 128,
        "image_size_cnn": list(IMG_SIZE_CNN),
        "false_accepts": len(false_accepts),
        "false_rejects": len(false_rejects),
    },
}

out_json = ARTIFACT_DIR / "baseline_results.json"
out_json.write_text(json.dumps(results, indent=2))
print("Saved:", out_json)

print("\n" + "=" * 62)
print(f"{'Metric':<22} {'NCC':>18} {'Siamese R18':>18}")
print("=" * 62)
for key in ["eer", "accuracy_at_eer", "far_at_eer", "frr_at_eer"]:
    print(f"{key:<22} {results['ncc_baseline'][key]:>18.4f} {results['siamese_resnet18_baseline'][key]:>18.4f}")
print("=" * 62)

Saved: /Users/spartan/Desktop/CS228/Project/artifacts/fvc2004/baseline_results.json

Metric                                NCC        Siamese R18
eer                                0.3084             0.0872
accuracy_at_eer                    0.6916             0.9128
far_at_eer                         0.3073             0.0865
frr_at_eer                         0.3094             0.0879
